# GPT-2 size capacity test

Tests loading and running forward passes through gpt2 (small/medium/large/xl) on M2 24GB / MPS.
Reports load time, forward-pass time, and process memory after each.

If a size fails (OOM or other error), we catch and continue.

In [ ]:
import gc
import time
import psutil
import torch
from transformer_lens import HookedTransformer

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'System memory: {psutil.virtual_memory().total / 1e9:.1f} GB')

PROMPT = 'The Eiffel Tower is located in the city of'
MODELS = ['gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl']

def proc_mem_gb():
    return psutil.Process().memory_info().rss / 1e9

def test_model(name):
    mem_before = proc_mem_gb()
    print(f'\n=== {name} ===')
    print(f'Process memory before: {mem_before:.2f} GB')
    try:
        t0 = time.time()
        model = HookedTransformer.from_pretrained(name, device=DEVICE)
        load_time = time.time() - t0
        mem_after_load = proc_mem_gb()
        n_params = sum(p.numel() for p in model.parameters())
        print(f'  load: {load_time:.1f}s | params: {n_params/1e6:.0f}M | mem: {mem_after_load:.2f} GB (+{mem_after_load-mem_before:.2f})')
        print(f'  cfg: d_model={model.cfg.d_model}, n_layers={model.cfg.n_layers}, d_mlp={model.cfg.d_mlp}, n_heads={model.cfg.n_heads}')
        tokens = model.to_tokens(PROMPT)
        t0 = time.time()
        with torch.no_grad():
            logits, cache = model.run_with_cache(tokens)
        fwd_time = time.time() - t0
        mem_after_fwd = proc_mem_gb()
        top5 = [model.tokenizer.decode([t.item()]) for t in logits[0, -1].topk(5).indices]
        print(f'  fwd+cache: {fwd_time:.2f}s | mem: {mem_after_fwd:.2f} GB | top-5: {top5}')
        del model, logits, cache, tokens
        gc.collect()
        if DEVICE == 'mps':
            torch.mps.empty_cache()
        print(f'  after cleanup: {proc_mem_gb():.2f} GB')
        return True
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {e}')
        gc.collect()
        if DEVICE == 'mps':
            torch.mps.empty_cache()
        return False

results = {name: test_model(name) for name in MODELS}
print('\n=== Summary ===')
for name, ok in results.items():
    print(f'  {name}: {"OK" if ok else "FAILED"}')

## Interpretation guide

- **OK across all four**: anything in the GPT-2 family is fair game for the work test.
- **OK through large, fails on xl**: stick with `gpt2-large` or smaller; gpt2-xl needs more memory than available.
- **Fails earlier**: try CPU instead of MPS, or reduce by trying float16 (not default in TransformerLens).
- **Slow forward passes (>5s for gpt2-small)**: something is off with MPS setup; consider falling back to CPU.

M2 24GB should comfortably handle all four. gpt2-xl is ~1.5B params (~6 GB in fp32); gpt2-large is ~774M (~3 GB).